# Predict on New Data with OctoPredictor

This notebook demonstrates how to use `OctoPredictor` to predict on new, unseen data after training an Octopus study. `OctoPredictor` ensembles the models from all outer splits into a single predictor that can be saved and loaded standalone for deployment.


## 1. Split Off Holdout Data and Train a Study

Before training, we set aside the last 100 rows of the dataset as "new unseen data" for prediction. The study is trained only on the remaining rows.

**Important:** This holdout set is **not** the same as the test set inside the nested cross-validation. The nested CV creates its own train/dev/test splits internally from the training data only. The holdout here simulates genuinely new data that the study has never seen during any stage of training, validation, or model selection -- as would happen in a real deployment scenario.


In [ ]:
import os
import tempfile

import pandas as pd

from octopus.example_data import load_breast_cancer_data
from octopus.modules import Octo
from octopus.study import OctoClassification
from octopus.types import ModelName

In [ ]:
# Load the breast cancer dataset
df, features, targets = load_breast_cancer_data()

print(f"Full dataset: {df.shape[0]} samples, {len(features)} features")
print(f"Classes: {targets}")
print(f"Target distribution: {df['target'].value_counts().sort_index().to_dict()}")

# Hold out the last 100 rows as "new unseen data"
n_holdout = 100
train_df = df.iloc[:-n_holdout].reset_index(drop=True)
new_df = df.iloc[-n_holdout:].reset_index(drop=True)

print(f"\nTraining set: {train_df.shape[0]} samples (used for study fitting)")
print(f"New data (held-out): {new_df.shape[0]} samples (simulates unseen data)")

In [ ]:
# Create and fit a classification study (2 outer folds, 5 trials for speed)
study = OctoClassification(
    study_name="predict_example",
    studies_directory=os.environ.get("STUDIES_PATH", "./studies"),
    target_metric="AUCROC",
    feature_cols=features,
    target_col="target",
    sample_id_col="index",
    stratification_col="target",
    n_outer_splits=2,
    n_cpus=1,
    workflow=[
        Octo(
            description="step1_octo",
            task_id=0,
            depends_on=None,
            models=[ModelName.ExtraTreesClassifier],
            n_trials=5,
            n_inner_splits=3,
        ),
    ],
)

study.fit(data=train_df)
print(f"Study saved to: {study.output_path}")

## 2. Load the Study


In [ ]:
from octopus.predict.study_io import load_study_info

study_info = load_study_info(study.output_path)

print(f"ML type: {study_info.config['ml_type']}")
print(f"Target metric: {study_info.config['target_metric']}")
print(f"Outer splits: {len(study_info.outer_split_dirs)}")
print(f"Features: {len(study_info.config['feature_cols'])}")

## 3. Create an OctoPredictor


In [ ]:
from octopus.predict import OctoPredictor

tp = OctoPredictor(study=study_info, task_id=0)
print(f"Loaded models for {len(tp._models)} outer splits")

## 4. Predict on New Data

Each outer-split model predicts independently, then results are averaged into an ensemble prediction. Use `per_split=True` to see individual split predictions.


In [ ]:
# Ensemble-averaged predictions (row_id + prediction)
predictions = tp.predict(new_df)
predictions.head(10)

In [ ]:
# With per-split predictions as additional columns
predictions_detail = tp.predict(new_df, per_split=True)
predictions_detail.head(10)

## 5. Predict Probabilities

For classification tasks, `predict_proba` returns class probabilities.


In [ ]:
# Ensemble-averaged probabilities (row_id + one column per class)
proba = tp.predict_proba(new_df)
proba.head(10)

In [ ]:
# With per-split probabilities as additional columns
proba_detail = tp.predict_proba(new_df, per_split=True)
proba_detail.head(10)

## 6. Evaluate Performance

Score each outer-split model on the held-out new data.


In [ ]:
perf = tp.performance(new_df, metrics=["AUCROC", "ACCBAL"])
perf

## 7. Save and Load for Deployment

Save the predictor to a standalone directory (models + metadata only, no data). Load it back and verify predictions match.


In [ ]:
# Save to a temporary directory
save_dir = tempfile.mkdtemp(prefix="octopus_predictor_")
tp.save(save_dir)
print(f"Saved predictor to: {save_dir}")
print(f"Contents: {os.listdir(save_dir)}")

In [ ]:
# Load the saved predictor (no study directory needed)
tp_loaded = OctoPredictor.load(save_dir)

# Verify predictions match
predictions_loaded = tp_loaded.predict(new_df)
pd.testing.assert_frame_equal(predictions, predictions_loaded)
print("Predictions match after save/load.")